# Técnica de eficiencia · Sistema final

**Qué se mide.** El sistema de M3 (agente de un paso + búsqueda híbrida + reranking + Qwen2.5-1.5B como decoder) corriendo
**en CPU con 2 hilos**, como en el hardware gratuito de despliegue, contra las versiones con la técnica de eficiencia:
cuantizar el decoder a GGUF (int8 e int4) y, en una última variante, cuantizar también el cross-encoder a int8.

Ejecutar en Colab con entorno **sin GPU** (CPU). Mismo eval set, mismo harness y mismo juez que M2 y M3. Hace falta
`ANTHROPIC_API_KEY` en los secretos de Colab.

## Umbral de aceptación (escrito antes de medir)

> Aceptamos perder hasta **1 acierto de 13** en el harness, sin bajar el recall@3 medio ni los aciertos en los casos
> adversariales (en particular los de abstención, M2-11 y M2-12), si la latencia **mediana** baja de **5 s**.

La versión cuantizada se compara contra la de M3 en la misma máquina y la misma sesión. Si no cumple, se dice.

## 0 · Entorno

In [ ]:
import os, subprocess, sys
from importlib.metadata import PackageNotFoundError, version as version_instalada
from pathlib import Path
from packaging.version import Version

REPO_URL = 'https://github.com/Bosnape/cabrejos-ortiz-valencia-lopez.git'
RAIZ = Path('/content/cabrejos-ortiz-valencia-lopez')
EN_COLAB = True
if not RAIZ.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(RAIZ)], check=True)
os.chdir(RAIZ)

try:
    torchao = version_instalada('torchao')
except PackageNotFoundError:
    torchao = None
if torchao is not None and Version(torchao) < Version('0.16.0'):
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)

pillow_fija = version_instalada('pillow')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers==6.0.1', 'transformers==5.16.1', 'anthropic==1.4.0', 'pandas==3.0.5',
    'python-dotenv==1.2.3', 'peft==0.20.0', 'accelerate==1.14.0', 'huggingface_hub',
    'chromadb', 'rank_bm25', 'pydantic', 'psutil', f'pillow=={pillow_fija}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python==0.3.36',
    '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cpu'], check=True)

import torch
print('Repositorio:', RAIZ)
print('Núcleos de CPU:', os.cpu_count(), '| GPU visible:', torch.cuda.is_available(),
      '(el sistema corre en CPU con 2 hilos aunque haya GPU)')

## 1 · La misma vara

Se reutilizan, sin modificarlas, las celdas del harness de `notebooks/rag/rag_avanzado.ipynb` (reglas de acierto, juez y
`harness`). El sistema evaluado es `final/sistema.py`.

In [ ]:
import json, math, re, time, unicodedata, gc, statistics
from datetime import datetime, timezone
import numpy as np
import pandas as pd

sys.path.insert(0, str(RAIZ / 'final'))
import sistema as S

base = json.loads((RAIZ / 'notebooks/rag/rag_avanzado.ipynb').read_text(encoding='utf-8'))
def ejecutar_celda_base(i):
    exec(compile(''.join(base['cells'][i]['source']), f'rag_avanzado.ipynb:celda_{i}', 'exec'), globals())

ejecutar_celda_base(2)                      # constantes, semillas, device

In [ ]:
ejecutar_celda_base(4)                      # cargar_eval_set, cargar_rubrica, cargar_corpus, validar_eval_set
eval_set = cargar_eval_set(RAIZ)
corpus = cargar_corpus(RAIZ)
assert validar_eval_set(eval_set)
for i in (18, 19, 20, 21, 22, 23):          # normas esperadas, acierto, juez, harness
    ejecutar_celda_base(i)
casos_agente = json.loads((RAIZ / 'final/casos_agente.json').read_text(encoding='utf-8'))
assert len(casos_agente) == 20
print(f'{len(eval_set)} casos, {len(corpus)} artículos, {len(casos_agente)} consultas de decisión del agente.')

## 2 · Las versiones

| Nombre | Decoder (también decide el agente) | Cross-encoder |
|---|---|---|
| `M3_transformers_fp32` | Qwen2.5-1.5B con transformers, float32 | fp32 |
| `gguf_fp16` | mismo modelo en GGUF fp16 (llama.cpp) | fp32 |
| `gguf_q8_0` | GGUF int8 | fp32 |
| `gguf_q4_k_m` | GGUF int4 | fp32 |
| `gguf_q4_k_m+ce_int8` | GGUF int4 | int8 dinámico |

El sistema de M3 corrió en una GPU T4 con pesos bf16. En CPU, bf16 no tiene kernels rápidos y haría la línea base
artificialmente lenta, así que "antes" es el mismo modelo en float32. `gguf_fp16` separa dos efectos: cambiar de motor
(transformers a llama.cpp) y cuantizar. Si `gguf_q8_0` y `gguf_q4_k_m` se comparan con `gguf_fp16`, solo cambia la cuantización.

In [ ]:
VARIANTES = {
    'M3_transformers_fp32':   dict(decoder='hf'),
    'gguf_fp16':              dict(decoder='fp16'),
    'gguf_q8_0':              dict(decoder='q8_0'),
    'gguf_q4_k_m':            dict(decoder='q4_k_m'),
    'gguf_q4_k_m+ce_int8':    dict(decoder='q4_k_m', ce_int8=True),
}
ANTES = 'M3_transformers_fp32'
A_CORRER = list(VARIANTES)          # para una prueba rápida: ['M3_transformers_fp32', 'gguf_q4_k_m']
REPETICIONES_LATENCIA = 0           # pasadas extra solo para latencia (sin juez); 0 = p50/p95 sobre los 13 casos
SALIDA = RAIZ / 'final/resultados'
SALIDA.mkdir(parents=True, exist_ok=True)

## 3 · Medición por versión

Para cada versión: carga, calentamiento (no cuenta), harness completo con el juez, latencia por componente, y las pruebas
que no miden exactitud: la **válvula** (¿el agente sigue diciendo "no busco" cuando corresponde y "busco" cuando es
laboral? 20 consultas, 10 y 10) y los **casos adversariales** del eval set.

La prueba de invarianza del M4 es visual (variar la imagen) y no aplica a este sistema, que no tiene componente visual.

In [ ]:
import psutil
proceso = psutil.Process()
mb = lambda b: round(b / 1e6)

def evaluar_variante(nombre, kwargs):
    gc.collect()
    rss0 = proceso.memory_info().rss
    t = time.perf_counter()
    s = S.Sistema(**kwargs).cargar()
    carga = time.perf_counter() - t
    rss_carga = proceso.memory_info().rss - rss0
    s.responder('hola'); s.historial.clear()                      # calentamiento

    res = harness(eval_set, s, juez, nombre=nombre, metadatos={'variante': nombre, **kwargs, 'hilos': s.hilos})
    hist = list(s.historial[-len(eval_set):])
    filas, detalle = pd.DataFrame(res['filas']), pd.DataFrame(res['detalle'])
    for k in ('agente_ms', 'hibrida_ms', 'reranker_ms', 'decoder_ms'):
        detalle[k] = [h.latencias_ms.get(k, math.nan) for h in hist]
    detalle['accion_agente'] = [h.accion_agente for h in hist]
    detalle['valvula'] = [h.valvula for h in hist]

    lat = list(filas['latencia_seg'])
    for _ in range(REPETICIONES_LATENCIA):
        for caso in eval_set:
            lat.append(s.responder(caso['input']).latencias_ms['total_ms'] / 1000)

    # valvula: solo decide el agente
    dec = []
    for c in casos_agente:
        d = s.decidir(c['pregunta'])
        dec.append({'id': c['id'], 'esperada': c['accion_esperada'], 'obtenida': d['action'], 'motivo': d['motivo'],
                    'intentos_formato': d['intentos_formato']})
    dec = pd.DataFrame(dec)
    dec['acierto'] = dec['esperada'] == dec['obtenida']

    adv = detalle[detalle['is_adversarial']]
    resumen = {
        'version': nombre,
        'aciertos': int(filas['acierto_dominio'].sum()), 'casos': len(filas),
        'recall_3_medio': round(filas['recall_3'].mean(), 3),
        'juez_medio': round(filas['juez_mitigado'].mean(), 2),
        'adversariales_acertados': int(adv['acierto_dominio'].sum()), 'adversariales': len(adv),
        'agente_correctas_de_20': int(dec['acierto'].sum()),
        'laborales_sin_buscar': int(((dec['esperada'] == S.NOMBRE_TOOL) & (dec['obtenida'] == 'no_tool')).sum()),
        'decisiones_invalidas': int((~dec['obtenida'].isin([S.NOMBRE_TOOL, 'no_tool'])).sum()),
        'p50_s': round(float(np.percentile(lat, 50)), 2), 'p95_s': round(float(np.percentile(lat, 95)), 2),
        'n_latencias': len(lat),
        'agente_s_mediana': round(float(detalle['agente_ms'].median()) / 1000, 2),
        'recuperacion_s_mediana': round(float((detalle['hibrida_ms'] + detalle['reranker_ms']).median()) / 1000, 2),
        'decoder_s_mediana': round(float(detalle['decoder_ms'].median()) / 1000, 2),
        'decoder_mb': mb(s.tamano_decoder_mb() * 1e6), 'cross_encoder_mb': mb(s.tamano_cross_encoder_mb() * 1e6),
        'rss_carga_mb_aprox': mb(rss_carga), 'carga_s': round(carga, 1),
    }
    detalle.to_csv(SALIDA / f'{nombre}_detalle.csv', index=False, lineterminator='\n')
    dec.to_csv(SALIDA / f'{nombre}_decisiones_agente.csv', index=False, lineterminator='\n')
    del s; gc.collect()
    return resumen, detalle, dec

resultados, detalles, decisiones = {}, {}, {}

In [ ]:
for nombre in A_CORRER:
    print('=' * 20, nombre)
    resultados[nombre], detalles[nombre], decisiones[nombre] = evaluar_variante(nombre, VARIANTES[nombre])
    pd.DataFrame(resultados.values()).to_csv(SALIDA / 'resumen_eficiencia.csv', index=False, lineterminator='\n')

## 4 · Tabla antes y después

In [ ]:
T = pd.DataFrame(resultados.values()).set_index('version')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', None)
display(T[['aciertos', 'recall_3_medio', 'juez_medio', 'adversariales_acertados', 'p50_s', 'p95_s', 'decoder_mb', 'cross_encoder_mb', 'rss_carga_mb_aprox']])
print('Latencia mediana por componente (s):')
display(T[['agente_s_mediana', 'recuperacion_s_mediana', 'decoder_s_mediana']])
print('Válvula (agente, 20 consultas):')
display(T[['agente_correctas_de_20', 'laborales_sin_buscar', 'decisiones_invalidas']])

In [ ]:
# Veredicto contra el umbral escrito antes de medir.
a = T.loc[ANTES]
filas_v = []
for nombre, r in T.iterrows():
    if nombre == ANTES: continue
    cond = {
        'perdida_aciertos<=1': (a['aciertos'] - r['aciertos']) <= 1,
        'recall_no_baja': r['recall_3_medio'] >= a['recall_3_medio'],
        'adversariales_no_bajan': r['adversariales_acertados'] >= a['adversariales_acertados'],
        'p50<5s': r['p50_s'] < 5,
    }
    filas_v.append({'version': nombre, **cond, 'se_cumple': all(cond.values()),
                    'aceleracion_p50': round(a['p50_s'] / r['p50_s'], 2),
                    'reduccion_tamano_decoder': round(a['decoder_mb'] / r['decoder_mb'], 2)})
V = pd.DataFrame(filas_v).set_index('version')
display(V)

In [ ]:
# Coincidencia de las decisiones del agente con las de la versión de M3 (la válvula puede cambiar al cuantizar).
base_dec = decisiones[ANTES].set_index('id')['obtenida']
for nombre, d in decisiones.items():
    if nombre == ANTES: continue
    igual = (d.set_index('id')['obtenida'] == base_dec).sum()
    print(f'{nombre}: {igual}/20 decisiones iguales a {ANTES}')

## 5 · Guardar y descargar

Los CSV quedan en `final/resultados/`. Descárguenlos y súbanlos al repositorio.

In [ ]:
import shutil
zip_ruta = shutil.make_archive('/content/resultados_eficiencia', 'zip', SALIDA)
from google.colab import files
files.download(zip_ruta)